In [ ]:
# === Mount Google Drive and install dependencies ===
from google.colab import drive
drive.mount("/content/drive")
!pip install -r /content/drive/MyDrive/iris/requirements.txt -q

# 03 — J3: Feature Inspection

**Purpose:** Determine whether the SAE learned interpretable, injection-sensitive
features — the true test of whether this project direction is viable.

**Context:** J2 did not meet its formal thresholds (reconstruction ratio 0.1082,
sparsity 14.4%), but the SAE captures 84% of activation variance. The question
is no longer "did we hit arbitrary numbers?" but "did the SAE learn features
that are functionally useful for injection detection?"

**Prerequisites:**
- Notebook 02 completed (SAE trained)
- `checkpoints/sae_d10240_lambda1e-04.pt` exists
- `checkpoints/j2_activations.npz` exists
- `data/processed/iris_dataset_balanced.json` exists

**J3 Pass Criterion (Design Document §6.1):**
For the top 20 most injection-sensitive features, examine top-10 activating
prompts. At least 5 features must show a clear, interpretable pattern.

**Outputs:**
- `results/figures/j3_sensitivity_distribution.png`
- `results/figures/j3_top_features_bar.png`
- `results/metrics/j3_feature_inspection.json`
- Feature dashboards printed inline for manual review

*Nathan Cheung (ncheung3@my.yorku.ca) | York University | CSSD 2221 | Winter 2026*

In [ ]:
# === Setup ===
import sys
import os

IN_COLAB = "google.colab" in sys.modules
PROJECT_ROOT = "/content/drive/MyDrive/iris" if IN_COLAB else os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

from src.utils.helpers import set_seed, get_device
set_seed(42)
device = get_device()

## Step 1: Load Saved Artifacts

We load three things from previous notebooks:
1. The balanced dataset (for prompt texts and labels)
2. The cached activations (for the layer we trained on)
3. The trained SAE checkpoint

In [ ]:
# === Load dataset ===
from src.data.dataset import IrisDataset

dataset = IrisDataset.load("data/processed/iris_dataset_balanced.json")
dataset.summary()

texts = dataset.texts
labels = dataset.labels

In [ ]:
# === Load activations ===
import numpy as np

data = np.load("checkpoints/j2_activations.npz")
# Load the layer that J2 trained on. Check j2_evaluation.json for
# which layer was used, or look at the J2 notebook output.
# We load all layers so we can pick the right one.
activations = {i: data[f'layer_{i}'] for i in range(36)}
saved_labels = data['labels']

# Verify labels match the dataset
assert np.array_equal(saved_labels, np.array(labels)), \
    "Label mismatch between saved activations and dataset!"
print(f"Loaded activations for {len(saved_labels)} examples across 36 layers")

In [ ]:
# === Load trained SAE ===
import torch
import json
from pathlib import Path
from src.sae.architecture import SparseAutoencoder

# Find the SAE checkpoint — look for the one with 10240 features
checkpoint_path = Path("checkpoints/sae_d10240_lambda1e-04.pt")
if not checkpoint_path.exists():
    # Fall back to scanning for any SAE checkpoint
    candidates = list(Path("checkpoints").glob("sae_*.pt"))
    print(f"Expected checkpoint not found. Available: {candidates}")
    checkpoint_path = candidates[0] if candidates else None
    assert checkpoint_path, "No SAE checkpoint found! Run notebook 02 first."

checkpoint = torch.load(checkpoint_path, map_location=device)
config = checkpoint['config']
print(f"Loaded checkpoint: {checkpoint_path}")
print(f"Config: {config}")

# Reconstruct the SAE with the same architecture
sae = SparseAutoencoder(
    d_input=config['d_input'],
    expansion_factor=config['expansion_factor'],
    sparsity_coeff=config.get('sparsity_coeff', 1e-4),
)
sae.load_state_dict(checkpoint['model_state_dict'])
sae = sae.to(device)
sae.eval()

# Read the training layer from J2's saved metrics so we use the
# same layer the SAE was actually trained on.
with open("results/metrics/j2_evaluation.json") as f:
    j2_metrics = json.load(f)
TARGET_LAYER = j2_metrics["train_layer"]
print(f"\nSAE: {sae.d_sae} features, trained on layer {TARGET_LAYER}")

## Step 2: Compute Feature Activations and Sensitivity Scores

For every prompt in the dataset, we run the SAE encoder to get the
10240-dimensional sparse feature vector. Then we compute each feature's
injection-sensitivity: the difference in mean activation between
injection and normal prompts.

```
sensitivity(feature_i) = mean_activation_on_injections(feature_i)
                        - mean_activation_on_normal(feature_i)
```

In [ ]:
# === Compute sparse feature activations for all prompts ===
from src.analysis.features import compute_feature_activations

# Use the activations from the layer the SAE was trained on
layer_acts = activations[TARGET_LAYER]
feature_matrix = compute_feature_activations(sae, layer_acts, device=device)

print(f"Feature matrix shape: {feature_matrix.shape}")
print(f"  (N={feature_matrix.shape[0]} prompts, d_sae={feature_matrix.shape[1]} features)")
print(f"  Mean active features per prompt: {(feature_matrix > 0).sum(axis=1).mean():.0f}")
print(f"  Sparsity: {(feature_matrix > 0).mean():.1%}")

In [ ]:
# === Compute injection-sensitivity scores ===
from src.analysis.features import compute_sensitivity_scores

sensitivity = compute_sensitivity_scores(feature_matrix, np.array(labels))

In [ ]:
# === Visualize the sensitivity distribution ===
# Most features should be near zero. Heavy tails indicate the SAE
# learned injection-relevant structure.
from src.analysis.features import plot_sensitivity_distribution

plot_sensitivity_distribution(
    sensitivity,
    save_path="results/figures/j3_sensitivity_distribution.png",
)

In [ ]:
# === Get and visualize the top 20 most sensitive features ===
from src.analysis.features import get_top_features, plot_top_features_bar

top_indices, top_values = get_top_features(sensitivity, k=20)

print("Top 20 features by |sensitivity|:")
for idx, val in zip(top_indices, top_values):
    direction = "injection" if val > 0 else "normal"
    print(f"  Feature {idx:4d}: sensitivity = {val:+.4f} ({direction}-associated)")

plot_top_features_bar(
    top_indices, top_values,
    save_path="results/figures/j3_top_features_bar.png",
)

## Step 3: Feature Dashboards — The J3 Test

This is the critical step. For each of the top 20 features, we look at
the 10 prompts that activate it most strongly. We're looking for:

- **Clear pattern**: Do the top-activating prompts share a common theme?
  (e.g., all contain "ignore previous instructions", or all are coding questions)
- **Class coherence**: Do injection-associated features mostly activate on
  injections? Do normal-associated features mostly activate on normal prompts?
- **Interpretability**: Can a human describe what the feature "detects"?

**J3 passes if at least 5 of the 20 features show clear, interpretable patterns.**

Read through each dashboard below and mentally note which features have
a clear pattern vs. which look like random noise.

In [ ]:
# === Print dashboards for all top 20 features ===
from src.analysis.features import get_top_activating_examples, print_feature_dashboard

for feat_idx, sens_val in zip(top_indices, top_values):
    top_examples = get_top_activating_examples(
        feature_matrix=feature_matrix,
        feature_idx=feat_idx,
        texts=texts,
        labels=labels,
        k=10,
    )
    print_feature_dashboard(feat_idx, sens_val, top_examples)

## Step 4: Quantitative Coherence Check

Beyond manual inspection, we can quantify how class-coherent each
feature is: for injection-associated features, what fraction of the
top-10 activating examples are actually injections? And vice versa.

In [ ]:
# === Compute class coherence for top features ===
# For each feature, what % of its top-10 activating examples match
# the expected class (injection for positive features, normal for negative)?
coherence_results = []

for feat_idx, sens_val in zip(top_indices, top_values):
    top_examples = get_top_activating_examples(
        feature_matrix, feat_idx, texts, labels, k=10
    )

    # Expected class based on sensitivity direction
    expected_label = 1 if sens_val > 0 else 0
    matches = sum(1 for ex in top_examples if ex['label'] == expected_label)
    coherence = matches / len(top_examples)

    coherence_results.append({
        'feature': int(feat_idx),
        'sensitivity': float(sens_val),
        'direction': 'injection' if sens_val > 0 else 'normal',
        'coherence': coherence,
        'matches': matches,
    })

# Print summary
print(f"{'Feature':>8} {'Sensitivity':>12} {'Direction':>10} {'Coherence':>10} {'Matches':>8}")
print('-' * 55)
for r in coherence_results:
    print(f"{r['feature']:>8d} {r['sensitivity']:>+12.4f} {r['direction']:>10} "
          f"{r['coherence']:>10.0%} {r['matches']:>5d}/10")

# Count features with strong coherence (>= 70% match)
n_coherent = sum(1 for r in coherence_results if r['coherence'] >= 0.7)
mean_coherence = np.mean([r['coherence'] for r in coherence_results])
print(f"\nFeatures with >= 70% class coherence: {n_coherent}/20")
print(f"Mean coherence across top 20: {mean_coherence:.0%}")

In [ ]:
# === Save J3 metrics ===
j3_results = {
    'target_layer': TARGET_LAYER,
    'd_sae': int(sae.d_sae),
    'n_features_inspected': 20,
    'n_coherent_70pct': n_coherent,
    'mean_coherence': float(mean_coherence),
    'features': coherence_results,
}

metrics_path = 'results/metrics/j3_feature_inspection.json'
os.makedirs(os.path.dirname(metrics_path), exist_ok=True)
with open(metrics_path, 'w') as f:
    json.dump(j3_results, f, indent=2, default=lambda x: float(x))
print(f'Saved to {metrics_path}')

In [ ]:
# === J3 Verdict ===
print('\n' + '=' * 60)
print('J3 FEATURE INSPECTION — VERDICT')
print('=' * 60)
print(f'Features inspected:          20')
print(f'Features with >= 70% coherence: {n_coherent}')
print(f'Mean coherence:              {mean_coherence:.0%}')
print()

# J3 passes if >= 5 features show clear, interpretable patterns.
# We use 70% coherence as a quantitative proxy for "interpretable":
# if 7+ of the top-10 examples match the expected class, the feature
# is capturing something real, not noise.
if n_coherent >= 5:
    print('J3 PASSED — at least 5 features show interpretable patterns.')
    print('\nThe SAE has learned injection-relevant structure.')
    print('Proceed to full experiments (C1-C4).')
else:
    print('J3 FAILED — fewer than 5 features show clear patterns.')
    print('\nConsider: different layer, larger dataset, or revised approach.')
print('=' * 60)
print()
print('NOTE: Also review the feature dashboards above manually.')
print('Coherence is a proxy — some features may capture subtle')
print('patterns that the 70% threshold misses.')

## Summary

This notebook completed:
1. **Feature activation extraction** — ran SAE encoder on all 1000 prompts
2. **Sensitivity scoring** — computed injection-sensitivity for all features
3. **Feature dashboards** — inspected top-10 activating examples for 20 features
4. **Coherence quantification** — measured class consistency of top features

**Junction decision:**
- J1: PASSED (Cohen's d up to 7.26 — strong centroid separation)
- J2: Partial (reconstruction above threshold, but functionally useful)
- J3: Review the dashboards above

**If J3 passed:** The project direction is viable. Proceed to scaling up
the dataset and running core experiments C1-C4.

**If J3 failed:** Consider training the SAE on a different layer, expanding
the dataset, or pivoting to Path A.